# 02 · B1 — Baseline chính thức Qwen2.5-VL-3B zero-shot trên L4 (W7)

**Vấn đề:** B0 (W5) chạy bằng **fp16**, mà W6–W7 đã chứng minh fp16 gây tràn số (23/2.000 câu ViTextVQA ra `!!!!`, tái hiện 23/23 trên cả T4 lẫn L4). Ngoài ra dự đoán bf16 chỉ trùng fp32 **78,1%** trên 73 câu probe. Vì vậy B0 **không dùng làm mốc so sánh** cho B2 được, phải chạy lại baseline bằng cấu hình đã chốt.

**B1 = mốc chính thức của đồ án.** Mọi cải tiến (B2 QLoRA, ablation) so với B1 dưới điều kiện y hệt:

| Điều kiện | Giá trị (chốt ở notebook 05) |
|---|---|
| Model | `Qwen/Qwen2.5-VL-3B-Instruct`, zero-shot, greedy, seed 42 |
| dtype | **bf16** (cả `torch_dtype` lẫn `bnb_4bit_compute_dtype`) |
| Lượng tử hoá | **4-bit NF4** + double quant (giống B2 QLoRA) |
| `max_pixels` | **401.408 (512 token ảnh)** |
| GPU | **NVIDIA L4** (cùng GPU với B2 để so được thời gian) |
| Test | ViVQA test 3.001 câu · ViTextVQA tập con 2.000 câu (seed 42) — **đã khoá từ commit 37da80a** |

**Nguyên tắc:** notebook không sửa config. Script `run_baseline.py` tự kiểm tra config đúng bản chuẩn W07 và ghi `run_signature.json`, nên không thể vô tình chạy tiếp một lô cũ với cấu hình khác.

**Thời gian ước tính (chưa đo trên L4 cho 2 tập này):** ViVQA ~35–50 phút · ViTextVQA 2.000 câu ~35–40 phút · (tuỳ chọn) ViTextVQA full test 10.028 câu ~2,5–3 giờ. Giữ máy bật, tắt Sleep.

**Nếu Colab rớt:** chạy lại từ phần 1 rồi phần 3 — các lô 500 câu đã xong tự bỏ qua, run đã xong hẳn cũng bỏ qua.

## 1. Thiết lập môi trường

Mount Drive, clone/pull code mới nhất, cài thư viện (giống hệt notebook 05: pin `transformers==5.17.0`), nạp `HF_TOKEN` nếu có. Cell **dừng hẳn** nếu GPU không phải L4, để B1 và B2 chắc chắn chạy cùng phần cứng.

*Kỳ vọng: commit ≥ `e0984e8`, `GPU: NVIDIA L4`, `✅ Thiết lập xong`. Khoảng 1–2 phút.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ B1 phải chạy trên L4 (cùng GPU với B2). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Danh sách run B1 + kiểm tra cấu hình

Tên run theo quy ước `W07_B1_<chế độ>_<model>_<dataset>_<dtype>_<GPU>_seed42`. ViTextVQA full test (10.028 câu) để **tắt mặc định** vì dài ~2,5–3 giờ: bật `RUN_FULL_TEST = True` khi có một buổi rảnh GPU.

*Kỳ vọng: bảng cấu hình in ra `bfloat16 | 401408 | nf4 | bfloat16`, và danh sách 2 run (3 nếu bật full test).*

In [ ]:
import json, yaml

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
CHUNK      = 500
RUN_FULL_TEST = False   # True: chạy thêm ViTextVQA full test 10.028 câu (~2,5–3 giờ)

B1_RUNS = {
    'vivqa':         {'dataset': 'vivqa', 'data_file': None,
                      'run': 'W07_B1_zeroshot_qwen25vl_vivqa_test_bf16_L4_seed42'},
    'vitextvqa2000': {'dataset': 'vitextvqa_official', 'data_file': None,
                      'run': 'W07_B1_zeroshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42'},
}
if RUN_FULL_TEST:
    B1_RUNS['vitextvqa_full'] = {'dataset': 'vitextvqa_official',
                                 'data_file': f'{DRIVE_ROOT}/data/vitextvqa_official/test.json',
                                 'run': 'W07_B1_zeroshot_qwen25vl_vitextvqa_testfull_bf16_L4_seed42'}

# B0 = W05 fp16 trên T4: chỉ để tham khảo, KHÔNG phải mốc so sánh
B0_RUNS = {'vivqa': 'W05_zeroshot_qwen25vl_vivqa_test_seed42',
           'vitextvqa2000': 'W05_zeroshot_qwen25vl_vitextvqa_test2000_seed42'}

cfg = yaml.safe_load(open('configs/qwen_lora.yaml', encoding='utf-8'))
mdl, q = cfg['model'], cfg['quantization']
print('torch_dtype | max_pixels | quant | compute dtype')
print(f"{mdl['torch_dtype']} | {mdl['max_pixels']} | {q.get('bnb_4bit_quant_type')} | {q.get('bnb_4bit_compute_dtype')}")
assert (mdl['torch_dtype'], mdl['max_pixels'], q.get('bnb_4bit_quant_type'), q.get('bnb_4bit_compute_dtype')) \
       == ('bfloat16', 401408, 'nf4', 'bfloat16'), 'Config KHÔNG phải bản chốt W07 -> dừng, báo Claude'
for k, r in B1_RUNS.items():
    print(f"  {k:15s} -> {r['run']}")

## 3. Chạy B1 (theo lô 500 câu, resume được)

Mỗi run: đưa ảnh về `/content` → nạp model (~2–4 phút) → chạy từng lô 500 câu, lưu ngay lên Drive → gộp và tính EM / VQA-Acc / ANLS + token-F1 (phụ) → ghi `metrics.json`, `predictions.json`, `run_signature.json`. Run nào đã có `metrics.json` thì bỏ qua (không nạp lại model).

*Kỳ vọng: với mỗi run, các dòng `[lô k/n] xong ...` hiện dần, cuối cùng `== KẾT QUẢ: EM=... | token-F1 (phụ)=...`. ViVQA 7 lô, ViTextVQA 4 lô. Giữ tab mở.*

In [ ]:
for key, r in B1_RUNS.items():
    run, ds = r['run'], r['dataset']
    extra = f"--data_file {r['data_file']}" if r['data_file'] else ''
    if os.path.exists(f'{EXP_ROOT}/{run}/metrics.json'):
        print(f'⏭ {key}: đã có metrics.json -> bỏ qua')
        continue
    print(f'\n▶ {key}: {run}')
    !python -u -m src.eval.run_baseline --dataset {ds} --run_name {run} --chunk {CHUNK} {extra}

## 4. Kết quả B1 (+ đối chiếu B0 chỉ để tham khảo)

Ba bảng:
1. **B1** trên từng tập: metric chính (EM, VQA-Acc, ANLS), token-F1 (phụ), số câu hỏng (rỗng hoặc chỉ toàn `!`), tốc độ, GPU, commit.
2. **B0 (fp16, T4) vs B1 (bf16, L4)** trên cùng câu test: token-F1 của cả hai được tính lại bằng cùng một hàm; đếm câu đổi đúng→sai và sai→đúng theo EM. Bảng này giải thích *vì sao* phải chạy lại baseline, **không** dùng để khẳng định cải tiến.
3. ViVQA tách theo loại câu hỏi (color / location / object / number).

*Kỳ vọng: B1 có **0 câu hỏng** trên cả hai tập (B0 ViTextVQA có 23).*

In [ ]:
import re
import pandas as pd
from src.eval.metrics import exact_match, token_f1

def load_run(run):
    d = f'{EXP_ROOT}/{run}'
    if not os.path.exists(f'{d}/metrics.json'):
        return None, None
    return (json.load(open(f'{d}/metrics.json', encoding='utf-8')),
            json.load(open(f'{d}/predictions.json', encoding='utf-8')))

def is_degenerate(p):
    return not re.sub(r'[!\s]', '', p or '')

rows, B1, B0 = [], {}, {}
for key, r in B1_RUNS.items():
    res, preds = load_run(r['run'])
    if res is None:
        print(f'⚠️ {key}: chưa có metrics.json (run lỗi hoặc chưa chạy)'); continue
    B1[key] = (res, preds)
    m, meta = res['metrics'], res['meta']
    rows.append({'run': key, 'n': meta['num_samples'], 'EM': m['exact_match'], 'VQA-Acc': m['vqa_accuracy'],
                 'ANLS': m['anls'], 'token-F1 (phụ)': m.get('token_f1'),
                 'câu hỏng': sum(is_degenerate(p['prediction']) for p in preds),
                 's/câu': meta['sec_per_sample'], 'phút': round(meta['inference_time_sec'] / 60, 1),
                 'GPU': str(meta['hardware'].get('gpus_seen')), 'dtype': meta['torch_dtype'],
                 'max_pixels': meta['max_pixels'], 'commit': meta['git_commit']})
print('==== Bảng 1. B1 — baseline chính thức (bf16, 512 tok, 4-bit NF4, L4) ====')
display(pd.DataFrame(rows).round(4))

cmp_rows = []
for key, b0_run in B0_RUNS.items():
    res0, p0 = load_run(b0_run)
    if res0 is None or key not in B1:
        continue
    B0[key] = (res0, p0)
    p1 = {p['question_id']: p for p in B1[key][1]}
    common = [p for p in p0 if p['question_id'] in p1]
    em0 = [exact_match(p['prediction'], p['answers']) for p in common]
    em1 = [exact_match(p1[p['question_id']]['prediction'], p['answers']) for p in common]
    for name, preds, em, mm in [('B0 fp16 T4 (W05)', common, em0, res0['metrics']),
                                ('B1 bf16 L4 (W07)', [p1[p['question_id']] for p in common], em1, B1[key][0]['metrics'])]:
        cmp_rows.append({'tập': key, 'run': name, 'n chung': len(common), 'EM': mm['exact_match'],
                         'ANLS': mm['anls'],
                         'token-F1 (tính lại)': sum(token_f1(p['prediction'], p['answers']) for p in preds) / len(preds),
                         'câu hỏng': sum(is_degenerate(p['prediction']) for p in preds)})
    same = sum(p['prediction'].strip() == p1[p['question_id']]['prediction'].strip() for p in common)
    print(f"{key}: dự đoán trùng nhau {same}/{len(common)} ({same/len(common):.1%}) | "
          f"EM đúng→sai {sum(bool(a) and not b for a, b in zip(em0, em1))} | sai→đúng {sum(bool(b) and not a for a, b in zip(em0, em1))}")
print('\n==== Bảng 2. B0 vs B1 — CHỈ THAM KHẢO (khác dtype + GPU) ====')
display(pd.DataFrame(cmp_rows).round(4))

if 'vivqa' in B1:
    types = ['color', 'location', 'object', 'number']
    tt = [{'loại': t, 'n': B1['vivqa'][0]['meta'].get('question_type_counts', {}).get(t),
           'B0 EM': (B0.get('vivqa', ({'metrics': {}},))[0]['metrics'].get(f'exact_match__{t}')),
           'B1 EM': B1['vivqa'][0]['metrics'].get(f'exact_match__{t}'),
           'B1 ANLS': B1['vivqa'][0]['metrics'].get(f'anls__{t}')} for t in types]
    print('\n==== Bảng 3. ViVQA theo loại câu hỏi ====')
    display(pd.DataFrame(tt).round(4))

## 5. Soi nhanh dự đoán

15 ví dụ ngẫu nhiên mỗi tập (seed 0), và **toàn bộ** câu hỏng nếu có. Đây là đầu vào cho error analysis + hallucination analysis ở W8, chưa phải phân tích chính thức.

*Kỳ vọng: không có mục "CÂU HỎNG".*

In [ ]:
import random
for key, (res, preds) in B1.items():
    print(f'\n######## {key} — 15 ví dụ ngẫu nhiên ########')
    for p in random.Random(0).sample(preds, min(15, len(preds))):
        ok = '✓' if exact_match(p['prediction'], p['answers']) else '✗'
        print(f"{ok} Q: {p['question']}\n    PRED: {p['prediction']!r}\n    GOLD: {p['answers'][:3]}")
    bad = [p for p in preds if is_degenerate(p['prediction'])]
    if bad:
        print(f'\n⚠️ CÂU HỎNG ({len(bad)}):', [p['question_id'] for p in bad][:30])

## 6. Lưu tóm tắt B1

Ghi `experiments/W07_B1_summary_L4_seed42/b1_summary.json` (Bảng 1–3 + điều kiện chạy). Cursor chép file này cùng `metrics.json` / `predictions.json` / `run_signature.json` của các run B1 vào repo (không chép `chunks/`).

*Kỳ vọng: `Ghi -> .../b1_summary.json`. Sau đó **Disconnect and delete runtime**.*

In [ ]:
OUT_DIR = f'{EXP_ROOT}/W07_B1_summary_L4_seed42'
os.makedirs(OUT_DIR, exist_ok=True)
summary = {
    'what': 'B1 = baseline chính thức (zero-shot) Qwen2.5-VL-3B-Instruct; mốc so sánh cho B2 và mọi ablation',
    'conditions': {'torch_dtype': 'bfloat16', 'quantization': '4-bit NF4 + double quant, compute bf16',
                   'max_pixels': 401408, 'decoding': 'greedy', 'seed': 42, 'gpu': GPU,
                   'test_locked_since_commit': '37da80a'},
    'runs': {k: r['run'] for k, r in B1_RUNS.items()},
    'table1_b1': rows,
    'table2_b0_vs_b1_reference_only': cmp_rows,
    'note': 'B0 (W05, fp16, T4) không so sánh trực tiếp được: fp16 tràn số; bảng 2 chỉ để giải thích lý do chạy lại.',
}
with open(f'{OUT_DIR}/b1_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=str)
print('Ghi ->', f'{OUT_DIR}/b1_summary.json')
print('➡ Xong: Runtime -> Disconnect and delete runtime, rồi kiểm tra Manage sessions.')